# Merge  firm (certero, 2024-hoy)  +  firm3 (extracción Python)  →  firm4

- **firm** manda: sus Mercados/Decisión/fechas son la verdad; donde esté vacío se completa con firm3.
- **firm3** aporta Empresas involucradas / Grupo y las columnas V2 (mercado/relaciones extraídos, referenciales).
- Se une por **clave TIPO-NÚMERO** (CONC-1951), robusta a `(PROSUM)`, puntos y espacios duros — NO por Nº de Resolución (que es NaN en PROSUM/OPI y hacía explotar el merge a 2217 filas).
- firm3 se **consolida**: un expediente = una fila (junta versiones CONC-1010 + CONC-1010_2).

In [7]:
import re, json
import numpy as np
import pandas as pd

In [8]:
firm = pd.read_excel(r"C:\Users\Admin\Documents\GitHub\buscadorANC\firm.xlsx")
dfs  = pd.read_excel(r"C:\Users\Admin\Documents\GitHub\buscadorANC\firmadas3.xlsx")
firm.columns = firm.columns.str.strip()
dfs.columns  = dfs.columns.str.strip()

## Clave robusta TIPO-NÚMERO (ignora PROSUM / puntos / espacios duros)

In [9]:
def clave(s):
    if pd.isna(s):
        return None
    m = re.search(r"(CONC|OPI|DP|INC)\D*(\d{3,4})", str(s).upper())
    return f"{m.group(1)}-{m.group(2)}" if m else None

firm["_k"] = firm["Carpeta"].map(clave)
dfs["_k"]  = dfs["Carpeta"].map(clave)

In [10]:
# firm3 aporta el mercado/relaciones extraídos como columnas V2 (referenciales)
dfs = dfs.rename(columns={"Mercados relevantes":   "mercado_relev_V2",
                          "Relaciones económicas": "relaciones_econ_V2"})

## Consolidar firm3: un expediente = una fila (junta CONC-1010 + CONC-1010_2)

In [11]:
def _first(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else np.nan

def _union_emp(s):
    """une las listas JSON de empresas de las versiones, sin duplicar."""
    arr, vistos, out = [], set(), []
    for v in s.dropna():
        try:
            arr += json.loads(v)
        except Exception:
            pass
    for e in arr:
        k = (e.get("rol"), (e.get("nombre") or "").strip().lower())
        if k[1] and k not in vistos:
            vistos.add(k); out.append(e)
    return json.dumps(out, ensure_ascii=False) if out else np.nan

agg = {c: (_union_emp if c == "Empresas involucradas" else _first)
       for c in dfs.columns if c != "_k"}
dfs_c = dfs[dfs["_k"].notna()].groupby("_k", as_index=False).agg(agg)
print("firm3:", len(dfs), "-> consolidado por clave:", len(dfs_c))

firm3: 16 -> consolidado por clave: 16


## Merge: firm manda; se completan SÓLO sus faltantes con firm3

In [12]:
firm4 = firm.merge(dfs_c, on="_k", how="left", suffixes=("", "_f3"))

# completar SÓLO faltantes de firm con el dato de firm3 (firm manda)
fill_desde = {
    "Mercados relevantes":   "mercado_relev_V2",
    "Relaciones económicas": "relaciones_econ_V2",
    "Decisión":              "Decisión_f3",
    "Fecha_Ingreso":         "Fecha_Ingreso_f3",
    "Fecha_firma":           "Fecha_firma_f3",
    "Número de Dictamen":    "Número de Dictamen_f3",
    "Carátula":              "Carátula_f3",
    "Excluible":             "Excluible_f3",
    "tipo":                  "tipo_f3",
}
for base, src in fill_desde.items():
    if base in firm4 and src in firm4:
        firm4[base] = firm4[base].where(firm4[base].notna(), firm4[src])

# descartar columnas auxiliares _f3
firm4 = firm4.drop(columns=[c for c in firm4.columns if c.endswith("_f3")])

# agregar filas de firm3 que firm NO cubre (pre-2024 y no matcheadas)
faltan = dfs_c[~dfs_c["_k"].isin(firm["_k"].dropna())].copy()
firm4 = pd.concat([firm4, faltan], ignore_index=True)

In [13]:
# limpieza final
firm4 = firm4.dropna(subset=["Carpeta"])
firm4 = firm4.drop_duplicates()          # saca filas 100% idénticas (duplicados propios de firm)
firm4 = firm4.drop(columns=["_k"])

# chequeo rápido: no debe explotar ni repetir carpetas de más
print("filas firm4:", len(firm4))
_d = firm4["Carpeta"].map(clave).dropna().value_counts()
_d = _d[_d > 1]
print("carpetas duplicadas (vienen de firm, revisar a mano):", dict(list(_d.items())))
firm4

filas firm4: 1040
carpetas duplicadas (vienen de firm, revisar a mano): {'INC-1663': 4, 'OPI-232': 2}


,tipo,Carpeta,Excluible,Carátula,Fecha_Ingreso,Fecha_firma,Decisión,Número de Resolución,Número de Dictamen,Mercados relevantes,Relaciones económicas,meses,DIAS,Grupo/Empresa,Empresas involucradas,mercado_relev_V2,relaciones_econ_V2
0,CONC,CONC-1348,NaN,NaN,NaN,2016-11-14,NaN,RESOL-2017-16-APN-SECC#MP,IF-2016-03279567-APN-CNDC#MP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CONC,CONC-1354,NaN,NaN,NaN,2016-11-15,NaN,RESOL-2017-6-APN-SECC#MP,IF-2016-03359536-APN-CNDC#MP,NaN,NaN,NaN,NaN,EXPERIENCIA ART S.A.,NaN,NaN,NaN
2,CONC,CONC-1086,NaN,NaN,NaN,2016-11-21,NaN,RESOL-2017-1-APN-SECC#MP,IF-2016-03594821-APN-CNDC#MP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CONC,CONC-1230,NaN,NaN,NaN,2016-11-21,NaN,RESOL-2017-13-APN-SECC#MP,IF-2016-03594891-APN-CNDC#MP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CONC,CONC-1089,NaN,NaN,NaN,2016-11-22,NaN,RESOL-2017-26-APN-SECC#MP,IF-2016-03626872-APN-CNDC#MP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1035,CONC,CONC-2031,NaN,NaN,NaN,2025-09-02,Autoriza Art.14 a) Ley 27.442,DISFC-2025-73-APN-CNDC#MEC,NaN,NaN,NaN,NaN,NaN,COMPAÑÍA GENERAL DE COMBUSTIBLES S.A. de una p...,NaN,NaN,Horizontal + vertical
1036,CONC,CONC-2104,NaN,"PAN AMERICAN ENERGY, S.L., SUC. ARG. Y TANGO E...",NaN,2026-08-10,NaN,RESFC-2026-49-APN-TDC#ANC,RE-2026-77038806-APN-SCE#ANC,NaN,NaN,NaN,NaN,"PAN AMERICAN ENERGY, S.L., SUC. ARG. Y TANGO E...",NaN,NaN,NaN
1037,CONC,CONC-2111,NaN,CHS DE ARGENTINA S.A.S/ NOTIFICACIÓN ART. 9 DE...,NaN,2026-08-14,NaN,RESFC-2026-50-APN-TDC#ANC,RE-2026-78194363-APN-SCE#ANC,NaN,NaN,NaN,NaN,CHS DE ARGENTINA S.A.,NaN,NaN,NaN
1038,CONC,CONC-2124,NaN,IRSA INVERSIONES Y REPRESENTACIONES SOCIEDAD A...,NaN,2026-08-04,Autoriza Art.14 a) Ley 27.442,DI-2026-82-APN-SCE#ANC,IF-2026-74916371-APN-DNCE#CNDC,NaN,NaN,NaN,NaN,IRSA INVERSIONES Y REPRESENTACIONES SOCIEDAD A...,"[{""rol"": ""comprador"", ""nombre"": ""CUYO S.A. en ...",NaN,Horizontal + Vertical + Conglomerado


In [14]:
firm4.to_excel("firm4.xlsx", index=False)

---
**Antes de correr:** regenerar `firm3_.xlsx` con el extractor (`python extraer_firmadas3.py` → renombrar la salida a `firm3_.xlsx`) para que traiga las empresas del FIX 7. Cerrá `firm4.xlsx` en Excel antes de escribir.

_Nota:_ las carpetas que quedan duplicadas provienen de duplicados propios de `firm.xlsx` (ej. `INC-1663`, `CONC-1951 (PROSUM)` cargados 2 veces) y filas basura como `"12"` — conviene limpiarlas en el archivo certero.

In [15]:
# (referencia) carga multi-año original, por si hay que reconstruir firm3 desde los res_firmadas_<año>.xlsx:
# dfs = {}
# for i in range(2026, 2017, -1):
#     dfs[i] = pd.read_excel(f"C:\\Users\\Admin\\Documents\\ANC\\descargas_cndc\\res_firmadas_{i}.xlsx", sheet_name="firmadas")
# df_all = pd.concat([df.assign(anio=i) for i, df in dfs.items()], ignore_index=True)